# PyTorch tensors: Applied patterns and advanced

**Solution notebook — Delta Drills #113**

Run the cells top-to-bottom to see the reference answer execute.


## Problem

Write a function solve(values, name) that returns an instance of a PyTorch tensor SUBCLASS called NamedArray: it behaves like a normal tensor built from `values` but carries an extra attribute `name` holding the given string. The name must survive operations that create views, such as slicing — implement __array_finalize__ so derived tensors inherit it.


In [ ]:
%pip install -q numpy torch --index-url https://download.pytorch.org/whl/cpu

## Reference solution


In [ ]:
import torch as t
class NamedTensor(t.Tensor):
    _name = "no name"

    @staticmethod
    def __new__(cls, data, name="no name"):
        obj = t.as_tensor(data).as_subclass(cls)
        obj._name = name
        return obj

    @property
    def name(self):
        return self._name

    @classmethod
    def __torch_function__(cls, func, types, args=(), kwargs=None):
        out = super().__torch_function__(func, types, args, kwargs or {})
        if isinstance(out, NamedTensor):
            src = next((a for a in args if isinstance(a, NamedTensor)), None)
            out._name = getattr(src, "_name", "no name")
        return out


def solve(values, name):
    return NamedTensor(values, name)


r = solve([1, 2, 3], "range")
print(r, r.name)
